# RootSignal vLLM GPU benchmark
Reproduce the published Tesla T4 streaming concurrency result. Select a GPU runtime first.

In [ ]:
import os, subprocess, sys, time, urllib.request
repo = '/content/rootsignal-bench'
subprocess.run(['git','clone','https://github.com/medhavee-upadhyaya/rootsignal-bench.git',repo],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','vllm'],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchaudio'],check=False)
print(subprocess.check_output('nvidia-smi --query-gpu=name,uuid,driver_version,memory.total --format=csv,noheader',shell=True,text=True))

In [ ]:
model='Qwen/Qwen2.5-0.5B-Instruct'
revision='7ae557604adf67be50417f59c2c2f167def9a775'
log=open('/content/vllm.log','w')
server=subprocess.Popen([sys.executable,'-m','vllm.entrypoints.openai.api_server','--model',model,'--revision',revision,'--dtype','half','--port','8001','--gpu-memory-utilization','0.80','--max-model-len','2048'],stdout=log,stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if urllib.request.urlopen('http://127.0.0.1:8001/v1/models',timeout=2).status == 200: break
    except Exception: time.sleep(2)
else: raise RuntimeError('vLLM readiness timeout')

In [ ]:
os.chdir(repo)
subprocess.run([sys.executable,'-m','benchmarks.inference','--model',model,'--model-revision',revision,'--hardware','NVIDIA Tesla T4 15360 MiB','--server','vllm-0.27.1','--dtype','float16','--requests','20','--concurrency-sweep','1,2,4,8','--warmup','2','--max-tokens','64','--output','/content/vllm-result.json'],check=True)
print(open('/content/vllm-result.json').read())